In [ ]:
"""
Shared classes and functions for the Ames Housing pipeline.

Both ames_housing_pipeline.py (training) and webapp/app.py (the local demo)
import from this file. This is what makes housing_pipeline.pkl loadable
outside the training script: joblib needs every custom class that went into
the pipeline to be importable from the SAME module path it was defined in
when the pipeline was fit. Keeping them here, in one place both scripts
import, is what guarantees that.
"""

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold
from category_encoders import MEstimateEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression

RANDOM_STATE = 0

QUAL_MAP = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

def mathematical_transforms(df):
    X = pd.DataFrame(index=df.index)
    X['LivLotRatio'] = df['GrLivArea'] / (df['LotArea'] + 1)
    X['Spaciousness'] = (df['1stFlrSF'] + df['2ndFlrSF']) / (df['TotRmsAbvGrd'] + 1)
    X['TotalSF'] = df['1stFlrSF'] + df['2ndFlrSF'] + df['TotalBsmtSF']
    return X

def counts(df):
    X = pd.DataFrame(index=df.index)
    porch_cols = ["WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"]
    X['PorchTypes'] = (df[porch_cols] > 0.0).sum(axis=1)
    X['TotalBaths'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
    return X

def house_age(df):
    X = pd.DataFrame(index=df.index)
    X['HouseAge'] = df['YrSold'] - df['YearBuilt']
    X['RemodelAge'] = df['YrSold'] - df['YearRemodAdd']
    return X

def quality_condition_interactions(df):
    X = pd.DataFrame(index=df.index)
    X['Qual_x_Cond'] = df['OverallQual'] * df['OverallCond']

    bsmt_q = df['BsmtQual'].fillna('None').map(QUAL_MAP).astype(float)
    bsmt_c = df['BsmtCond'].fillna('None').map(QUAL_MAP).astype(float)
    X['Bsmt_Qual_x_Cond'] = bsmt_q * bsmt_c

    gar_q = df['GarageQual'].fillna('None').map(QUAL_MAP).astype(float)
    gar_c = df['GarageCond'].fillna('None').map(QUAL_MAP).astype(float)
    X['Garage_Qual_x_Cond'] = gar_q * gar_c
    return X

def sqrt_area_transforms(df):
    X = pd.DataFrame(index=df.index)
    area_cols = ['GrLivArea', 'TotalBsmtSF', 'LotArea', '1stFlrSF', '2ndFlrSF', 'GarageArea']
    for col in area_cols:
        X[f'Sqrt_{col}'] = np.sqrt(df[col].clip(lower=0))
    return X

def log_transforms(df):
    X = pd.DataFrame(index=df.index)
    skewed_cols = ['LotArea', 'LotFrontage', 'MasVnrArea', 'OpenPorchSF', 'WoodDeckSF']
    for col in skewed_cols:
        val = df[col].fillna(0).clip(lower=0)
        X[f'Log_{col}'] = np.log1p(val)
    return X

def numeric_categorical_interactions(df):
    X = pd.DataFrame(index=df.index)
    bsmt_qual_num = df['BsmtQual'].fillna('None').map(QUAL_MAP).astype(float)
    X['Bsmt_Area_x_Qual'] = df['TotalBsmtSF'].fillna(0) * bsmt_qual_num

    garage_qual_num = df['GarageQual'].fillna('None').map(QUAL_MAP).astype(float)
    X['Garage_Area_x_Qual'] = df['GarageArea'].fillna(0) * garage_qual_num
    return X

def pca_inspired_features(df):
    X = pd.DataFrame(index=df.index)
    X['Remod_x_BsmtArea'] = df['YearRemodAdd'] * df['TotalBsmtSF']
    return X

def indicate_outliers(df):
    X = pd.DataFrame(index=df.index)
    is_edwards = (df['Neighborhood'] == "Edwards")
    is_partial = (df['SaleCondition'] == "Partial")
    X['Is_Outlier_Case'] = (is_edwards & is_partial).astype(int)
    return X


class StaticFeatureEngineer(BaseEstimator, TransformerMixin):
    """Bundles every feature function above. Stateless -> nothing to leak."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        Xo = pd.DataFrame(index=X.index)
        Xo = Xo.join(mathematical_transforms(X))
        Xo = Xo.join(counts(X))
        Xo = Xo.join(house_age(X))
        Xo = Xo.join(quality_condition_interactions(X))
        Xo = Xo.join(sqrt_area_transforms(X))
        Xo = Xo.join(log_transforms(X))
        Xo = Xo.join(numeric_categorical_interactions(X))
        Xo = Xo.join(pca_inspired_features(X))
        Xo = Xo.join(indicate_outliers(X))
        return Xo


# -----------------------------------------------------------------------
# Cell: stateful feature transformers
# (each .fit() learns only from the training rows it's given — the bug fix)
# -----------------------------------------------------------------------
class NeighborhoodStatsFeatures(BaseEstimator, TransformerMixin):
    """Replaces the old neighborhood_advanced_stats(), which read the global
    X_full instead of its own argument. Now fit() learns the per-neighborhood
    GrLivArea mean/median/std from whatever training split it's given."""
    def __init__(self, group_col='Neighborhood', value_col='GrLivArea'):
        self.group_col = group_col
        self.value_col = value_col

    def fit(self, X, y=None):
        stats = X.groupby(self.group_col)[self.value_col].agg(['mean', 'median', 'std']).fillna(0)
        self.mean_ = stats['mean']
        self.median_ = stats['median']
        self.std_ = stats['std'].replace(0, 1)
        self.global_median_ = X[self.value_col].median()
        self.global_mean_ = X[self.value_col].mean()
        return self

    def transform(self, X):
        Xo = pd.DataFrame(index=X.index)
        med = X[self.group_col].map(self.median_).fillna(self.global_median_)
        mean = X[self.group_col].map(self.mean_).fillna(self.global_mean_)
        std = X[self.group_col].map(self.std_).fillna(1)
        Xo['Diff_From_Nhbd_Median'] = X[self.value_col] - med
        Xo['ZScore_GrLivArea_Nhbd'] = (X[self.value_col] - mean) / std
        return Xo


class ClusterFeatures(BaseEstimator, TransformerMixin):
    """KMeans is now fit inside .fit() on the training fold only."""
    def __init__(self, features, n_clusters=10, random_state=RANDOM_STATE):
        self.features = features
        self.n_clusters = n_clusters
        self.random_state = random_state

    def fit(self, X, y=None):
        self.scaler_ = StandardScaler()
        Xs = self.scaler_.fit_transform(X[self.features].fillna(0))
        self.kmeans_ = KMeans(n_clusters=self.n_clusters, n_init=20, random_state=self.random_state)
        self.kmeans_.fit(Xs)
        return self

    def transform(self, X):
        Xs = self.scaler_.transform(X[self.features].fillna(0))
        Xo = pd.DataFrame(index=X.index)
        Xo['Cluster'] = self.kmeans_.predict(Xs).astype(str)
        distances = self.kmeans_.transform(Xs)
        for i in range(distances.shape[1]):
            Xo[f'Centroid_Dist_{i}'] = distances[:, i]
        return Xo


class PCAFeatures(BaseEstimator, TransformerMixin):
    """PCA is now fit inside .fit() on the training fold only."""
    def __init__(self, features, n_components=4, random_state=RANDOM_STATE):
        self.features = features
        self.n_components = n_components
        self.random_state = random_state

    def fit(self, X, y=None):
        self.scaler_ = StandardScaler()
        Xs = self.scaler_.fit_transform(X[self.features].fillna(0))
        self.pca_ = PCA(n_components=self.n_components, random_state=self.random_state)
        self.pca_.fit(Xs)
        return self

    def transform(self, X):
        Xs = self.scaler_.transform(X[self.features].fillna(0))
        comps = self.pca_.transform(Xs)
        Xo = pd.DataFrame(index=X.index)
        for i in range(comps.shape[1]):
            Xo[f'PC_{i+1}'] = comps[:, i]
        return Xo


class CrossFoldTargetEncoder(BaseEstimator, TransformerMixin):
    """Same idea as the notebook's original CrossFoldEncoder (out-of-fold
    target encoding so training rows don't see their own target), but now
    scoped correctly: fit_transform() is what the outer Pipeline calls during
    .fit(), so the OOF split happens fresh inside every outer CV fold instead
    of once globally."""
    def __init__(self, cols=('Neighborhood',), m=1, n_splits=5, random_state=RANDOM_STATE):
        self.cols = list(cols)
        self.m = m
        self.n_splits = n_splits
        self.random_state = random_state

    def fit(self, X, y):
        # Encoder used at predict-time on genuinely new/unseen rows.
        self.full_encoder_ = MEstimateEncoder(cols=self.cols, m=self.m)
        self.full_encoder_.fit(X[self.cols], y)
        return self

    def fit_transform(self, X, y=None):
        self.fit(X, y)
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        out = pd.Series(index=X.index, dtype=float)
        for idx_fit, idx_holdout in kf.split(X):
            enc = MEstimateEncoder(cols=self.cols, m=self.m)
            enc.fit(X.iloc[idx_fit][self.cols], y.iloc[idx_fit])
            out.iloc[idx_holdout] = enc.transform(X.iloc[idx_holdout][self.cols])[self.cols[0]].values
        return pd.DataFrame({f'{self.cols[0]}_Encoded': out}, index=X.index)

    def transform(self, X):
        enc = self.full_encoder_.transform(X[self.cols])
        return pd.DataFrame({f'{self.cols[0]}_Encoded': enc[self.cols[0]].values}, index=X.index)


cluster_features = ["LotArea", "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "GrLivArea"]
pca_features = ["GarageArea", "YearRemodAdd", "TotalBsmtSF", "GrLivArea"]


class FullFeatureEngineer(BaseEstimator, TransformerMixin):
    """Everything above, composed into one transformer. Its .fit()/.transform()
    are what the outer Pipeline calls, so every sub-transformer only ever sees
    the current fold's training rows when it fits."""

    def fit(self, X, y=None):
        self.static_ = StaticFeatureEngineer().fit(X)
        self.nhbd_stats_ = NeighborhoodStatsFeatures().fit(X)
        self.cluster_ = ClusterFeatures(cluster_features).fit(X)
        self.pca_feats_ = PCAFeatures(pca_features).fit(X)
        self.nhbd_encoder_ = CrossFoldTargetEncoder(cols=['Neighborhood'], m=1)
        self.nhbd_encoder_.fit(X, y)
        return self

    def transform(self, X):
        Xo = X.copy()
        Xo = Xo.join(self.static_.transform(X))
        Xo = Xo.join(self.nhbd_stats_.transform(X))
        Xo = Xo.join(self.cluster_.transform(X))
        Xo = Xo.join(self.pca_feats_.transform(X))
        Xo = Xo.join(self.nhbd_encoder_.transform(X))
        return Xo

    def fit_transform(self, X, y=None):
        self.static_ = StaticFeatureEngineer().fit(X)
        self.nhbd_stats_ = NeighborhoodStatsFeatures().fit(X)
        self.cluster_ = ClusterFeatures(cluster_features).fit(X)
        self.pca_feats_ = PCAFeatures(pca_features).fit(X)
        self.nhbd_encoder_ = CrossFoldTargetEncoder(cols=['Neighborhood'], m=1)
        nhbd_encoded = self.nhbd_encoder_.fit_transform(X, y)

        Xo = X.copy()
        Xo = Xo.join(self.static_.transform(X))
        Xo = Xo.join(self.nhbd_stats_.transform(X))
        Xo = Xo.join(self.cluster_.transform(X))
        Xo = Xo.join(self.pca_feats_.transform(X))
        Xo = Xo.join(nhbd_encoded)
        return Xo


class PositiveMutualInfoSelector(BaseEstimator, TransformerMixin):
    """Replaces the manual `mi_series[mi_series > 0]` step. Because this is a
    transformer inside the Pipeline, mutual_info_regression is now computed
    fresh on each fold's training data only, instead of once on all of X."""
    def __init__(self, random_state=RANDOM_STATE):
        self.random_state = random_state

    def fit(self, X, y):
        mi = mutual_info_regression(X, y, random_state=self.random_state)
        self.support_ = mi > 0.0
        self.mi_scores_ = mi  # kept for the metadata dump below, not used for modeling
        return self

    def transform(self, X):
        if hasattr(X, 'iloc'):
            return X.iloc[:, self.support_]
        return X[:, self.support_]


# -----------------------------------------------------------------------
# Cell: column selectors for the ColumnTransformer
# Passed as callables so they're evaluated per-fold on the engineered
# training data, not pre-computed once on the full dataset.
# -----------------------------------------------------------------------
def numeric_selector(X):
    return [c for c in X.columns if X[c].dtype in ['int64', 'float64']]

def categorical_selector(X):
    # was `< 10`, which silently excluded the 10-category Cluster column
    return [c for c in X.columns if X[c].dtype == 'object' and X[c].nunique() <= 10]

In [ ]:
# =============================================================================
# Ames Housing — leak-free version of the Kaggle competition pipeline, with a full-feature engineering, 
# PCA, K-Means, and XGBoost model. This is the training script that produces the final fitted pipeline artifact 
# (housing_pipeline.pkl) and the metadata.json file for the web dashboard.
# =============================================================================

import joblib
import optuna
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import mutual_info_regression

# Every custom transformer (FullFeatureEngineer, ClusterFeatures, PCAFeatures,
# PositiveMutualInfoSelector, etc.) lives in pipeline_classes.py, and NOT in
# this file. That's what lets housing_pipeline.pkl be loaded later by the
# webapp/app.py demo (or anywhere else) without re-running training — joblib
# needs these classes importable from the same module path they were defined
# in, and pipeline_classes.py is that shared, stable path.
# from pipeline_classes import (
#     RANDOM_STATE,
#     FullFeatureEngineer,
#     PCAFeatures,
#     PositiveMutualInfoSelector,
#     numeric_selector,
#     categorical_selector,
#     pca_features,
# )

target_name = 'SalePrice'
train_url = '../input/train.csv'
test_url = '../input/test.csv'

# -----------------------------------------------------------------------
# Cell: load data
# -----------------------------------------------------------------------
X_full = pd.read_csv(train_url, index_col='Id')
X_test_full = pd.read_csv(test_url, index_col='Id')

X_full.dropna(axis=0, subset=[target_name], inplace=True)
y = X_full[target_name].copy()
X_full.drop([target_name], axis=1, inplace=True)

# -----------------------------------------------------------------------
# Cell: missing-value report (unchanged — purely descriptive)
# -----------------------------------------------------------------------
def missing_value_report(df, label):
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    report = pd.DataFrame({
        'Missing Values': missing,
        'Percentage (%)': (missing / len(df)) * 100
    }).sort_values(by='Missing Values', ascending=False)
    print(f"Missing values table for {label}")
    print("-" * 40)
    print(report)

missing_value_report(X_full, "training data")
print("=" * 40)
missing_value_report(X_test_full, "test data")

# -----------------------------------------------------------------------
# Feature engineering classes, the column selectors, and the stateless
# helper functions (mathematical_transforms, counts, etc.) now live in
# pipeline_classes.py and were imported above. Kept in one shared module so
# webapp/app.py can joblib.load() the fitted pipeline without duplicating
# any of this code.
# -----------------------------------------------------------------------


preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='mean', add_indicator=True), numeric_selector),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), categorical_selector),
    ]
)

# -----------------------------------------------------------------------
# Cell: the single end-to-end pipeline
# Everything — feature engineering, clustering, PCA, target encoding,
# imputation, one-hot, MI selection, model — lives in one object. This is
# what cross_val_score / Optuna fit per fold, and what gets saved at the end.
# -----------------------------------------------------------------------
full_pipeline = Pipeline(steps=[
    ('features', FullFeatureEngineer()),
    ('preprocess', preprocessor),
    ('mi_select', PositiveMutualInfoSelector(random_state=RANDOM_STATE)),
    ('model', XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
])

print("Pipeline assembled — feature engineering, clustering, PCA, target")
print("encoding, and MI selection are all fold-scoped now.")

# -----------------------------------------------------------------------
# Cell (optional, exploratory only): PCA variance / correlation plots
# These fit a throwaway copy of the PCA step on the FULL training set purely
# to visualize it — they are not part of the modeling pipeline above and
# don't affect the CV score or the final model.
# -----------------------------------------------------------------------
def plot_variance(pca, width=8, dpi=100):
    fig, axs = plt.subplots(1, 2)
    n = pca.n_components_
    grid = np.arange(1, n + 1)

    evr = pca.explained_variance_ratio_
    axs[0].bar(grid, evr, color='skyblue', edgecolor='black')
    axs[0].set(xlabel="Component", title="% Explained Variance", ylim=(0.0, 1.0))
    axs[0].set_xticks(grid)
    axs[0].grid(axis='y', linestyle='--', alpha=0.7)

    cv = np.cumsum(evr)
    axs[1].plot(np.r_[0, grid], np.r_[0, cv], "o-", color='crimson')
    axs[1].set(xlabel="Component", title="Cumulative Variance", ylim=(0.0, 1.0))
    axs[1].set_xticks(np.r_[0, grid])
    axs[1].grid(linestyle='--', alpha=0.7)

    fig.set(figwidth=width, dpi=dpi)
    plt.tight_layout()
    plt.show()

_diagnostic_pca_features = PCAFeatures(pca_features).fit(X_full)
print("--- The variance ratio explained by PCA (exploratory, full-data fit) ---")
plot_variance(_diagnostic_pca_features.pca_)

def corrplot(df, method="pearson", annot=True, **kwargs):
    corr = df.corr(method=method, numeric_only=True)
    sns.clustermap(corr, vmin=-1.0, vmax=1.0, cmap="icefire", method="complete", annot=annot, **kwargs)
    plt.title("Clustered Correlation Matrix", pad=15)
    plt.show()

print("--- Link map of PCA features with price (exploratory) ---")
pca_analysis_df = X_full[pca_features].copy()
pca_analysis_df['SalePrice'] = y
corrplot(pca_analysis_df, annot=True, figsize=(8, 8))

loadings = pd.DataFrame(
    _diagnostic_pca_features.pca_.components_.T,
    columns=[f'PC{i+1}' for i in range(_diagnostic_pca_features.pca_.n_components_)],
    index=pca_features,
)
print("\n--- PCA Loadings Table (exploratory) ---")
print(loadings.round(3))

# -----------------------------------------------------------------------
# Cell (optional, exploratory only): Mutual Information scores
# Fits FullFeatureEngineer + preprocessor once on the FULL training set,
# purely to plot which engineered features look informative. This is NOT
# what the model actually uses for selection — that's PositiveMutualInfoSelector
# inside full_pipeline, which recomputes MI fold-by-fold during CV/Optuna and
# final fit. This plot is only for you to look at.
# -----------------------------------------------------------------------
print('--- Calculating & Plotting Mutual Information (exploratory, full-data fit) ---')

_diagnostic_features = FullFeatureEngineer().fit_transform(X_full, y)
_diagnostic_preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='mean', add_indicator=True), numeric_selector),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), categorical_selector),
    ]
)
_diagnostic_transformed = _diagnostic_preprocessor.fit_transform(_diagnostic_features)
_diagnostic_feature_names = _diagnostic_preprocessor.get_feature_names_out()

mi_scores = mutual_info_regression(_diagnostic_transformed, y, random_state=RANDOM_STATE)
mi_series = pd.Series(mi_scores, index=_diagnostic_feature_names).sort_values(ascending=True)

plt.figure(figsize=(10, round(mi_series.count() * 0.3)), dpi=100)
mi_series.plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('Mutual Information Scores (exploratory)', fontsize=12, pad=15)
plt.xlabel('MI Score', fontsize=10)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("Top Feature Scores:")
print(mi_series.sort_values(ascending=False))

# -----------------------------------------------------------------------
# Cell: hyperparameter search
# cross_val_score now fits the WHOLE pipeline (features -> preprocess ->
# MI select -> model) inside each fold, so the reported MAE reflects what
# you'd actually see on unseen data.
# -----------------------------------------------------------------------
def objective(trial):
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 7),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'model__subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'model__colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'model__gamma': trial.suggest_float('gamma', 0.0, 1.0),
        'model__reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0),
    }
    full_pipeline.set_params(**params)

    scores = -1 * cross_val_score(
        full_pipeline, X_full, y, cv=5, scoring='neg_mean_absolute_error', n_jobs=1
    )
    return scores.mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='minimize')

print("The search is on for the best parameters (leak-free this time)...")
study.optimize(objective, n_trials=40)

print("\n" + "=" * 40)
print(f"Best CV MAE: ${study.best_value:,.2f}")
print("Best Parameters:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")
print("=" * 40)

# -----------------------------------------------------------------------
# Cell: final fit + predict + save
# The saved artifact is now the ENTIRE pipeline — no separate scaler,
# kmeans, pca, encoder, or ColumnTransformer objects to track by hand.
# -----------------------------------------------------------------------
best_params = {f'model__{k}': v for k, v in study.best_params.items()}
full_pipeline.set_params(**best_params)
full_pipeline.fit(X_full, y)

test_preds = full_pipeline.predict(X_test_full)
print("Pipeline fit completed successfully")

id_cols = [col for col in X_test_full.columns if col.lower() == 'id']
test_id = X_test_full[id_cols[0]] if id_cols else X_test_full.index

output = pd.DataFrame({'Id': test_id, target_name: test_preds})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

joblib.dump(full_pipeline, 'housing_pipeline.pkl')
print("Full pipeline exported successfully! Load it with joblib.load() and")
print("call .predict(raw_dataframe) directly — no other artifacts needed.")

# -----------------------------------------------------------------------
# Cell: export dashboard metadata (for the local web demo)
# Pulls numbers out of the already-fitted final pipeline — no extra fitting,
# no leakage. Written as plain JSON so a small Flask app can read it without
# needing scikit-learn installed.
# -----------------------------------------------------------------------
import json

fitted_preprocess = full_pipeline.named_steps['preprocess']
fitted_mi_select = full_pipeline.named_steps['mi_select']
fitted_model = full_pipeline.named_steps['model']
fitted_pca = full_pipeline.named_steps['features'].pca_feats_.pca_

all_feature_names = fitted_preprocess.get_feature_names_out()
selected_feature_names = all_feature_names[fitted_mi_select.support_]
selected_mi_scores = fitted_mi_select.mi_scores_[fitted_mi_select.support_]

importances = pd.Series(fitted_model.feature_importances_, index=selected_feature_names)
mi_scores_selected = pd.Series(selected_mi_scores, index=selected_feature_names)

metadata = {
    'best_cv_mae': float(study.best_value),
    'best_params': study.best_params,
    'n_features_total': int(len(all_feature_names)),
    'n_features_selected': int(len(selected_feature_names)),
    'top_feature_importances': [
        {'name': n, 'value': float(v)}
        for n, v in importances.sort_values(ascending=False).head(15).items()
    ],
    'top_mi_scores': [
        {'name': n, 'value': float(v)}
        for n, v in mi_scores_selected.sort_values(ascending=False).head(15).items()
    ],
    'pca_explained_variance_ratio': [float(v) for v in fitted_pca.explained_variance_ratio_],
    'train_rows': int(len(X_full)),
    'test_rows': int(len(X_test_full)),
}

with open('metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# A handful of real rows the web demo can let the user pick from and tweak,
# instead of asking a visitor to fill in ~80 raw columns by hand.
sample_houses = X_full.head(8).reset_index()
sample_houses.to_json('sample_houses.json', orient='records', indent=2)

print("Saved metadata.json and sample_houses.json for the local web dashboard.")